In [ ]:
using Bessels
using ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz

ω_b = ω_b_0 #+ Lamb_shift*2*pi # Shifted frequency of mode b

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ=1/10; # 1/ns from Leon's paper

ω_d=ω_b+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d # Pump Frequency

ϵ_0 = 0.25
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2

κ_2 = 4g2^2/κ;

# First basic approach, rough estimation of the Lyapunov exponent in a random case

In [ ]:
n=3000

bath=Exp_decay
bath_params =(ω_0=50*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
t_max = 40/κ_2

In [ ]:
1/κ

In [ ]:
t_max = 20/κ_2


ϕ_a,ϕ_b,q_a,q_b,t_list = two_photon_1d_chain(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 =0,κ=κ, ϵ_d=0, ω_d=0),
     init_mode_a=(ϕ_a_0=1.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=1.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=false)

In [ ]:
plot(t_list*κ,ϕ_b)

In [ ]:
F,freqs=FFT_perso(ϕ_b, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
Lamb_shift = f_shifted - ω_b/(2*pi)

In [ ]:
t_max=20/κ_2
ϕ_a_1,q_a_1,t_list = two_photon_1d_chain(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 =ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0+Lamb_shift*2*pi),
     init_mode_a=(ϕ_a_0=0.001,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=true,emission=false)

In [ ]:
Re_a_1 = ϕ_a_1 / sqrt(2*Z_a)
Im_a_1 = q_a_1 * sqrt(Z_a/2)

ϕ_rotating_a_1 =@. ϕ_a_1 * cos(ω_a * t_list) + q_a_1*Z_a* sin(-ω_a * t_list)
q_rotating_a_1 =@. -ϕ_a_1/Z_a * sin(-ω_a * t_list) + q_a_1* cos(ω_a * t_list)

Re_rotating_a_1 = ϕ_rotating_a_1 / sqrt(2*Z_a)
Im_rotating_a_1 = q_rotating_a_1 * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a_1,Im_a_1)
p2=plot(Re_rotating_a_1,Im_rotating_a_1)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("cat_state_chain.pdf")

In [ ]:
ϕ_a_2,q_a_2,t_list = two_photon_1d_chain(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 =ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0+Lamb_shift*2*pi),
     init_mode_a=(ϕ_a_0=0.00100001,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=true,emission=false)

In [ ]:
Re_a_2 = ϕ_a_2 / sqrt(2*Z_a)
Im_a_2 = q_a_2 * sqrt(Z_a/2)

ϕ_rotating_a_2 =@. ϕ_a_2 * cos(ω_a * t_list) + q_a_2*Z_a* sin(-ω_a * t_list)
q_rotating_a_2 =@. -ϕ_a_2/Z_a * sin(-ω_a * t_list) + q_a_2* cos(ω_a * t_list)

Re_rotating_a_2 = ϕ_rotating_a_2 / sqrt(2*Z_a)
Im_rotating_a_2 = q_rotating_a_2 * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a_2,Im_a_2)
p2=plot(Re_rotating_a_2,Im_rotating_a_2)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("cat_state_chain.pdf")

In [ ]:
diff_re = Re_rotating_a_2 .- Re_rotating_a_1
diff_im = Im_rotating_a_2 .- Im_rotating_a_1

p1 = plot(t_list*κ_2, diff_re, label=L"\Re(a_2-a_1)")
plot!(t_list*κ_2, diff_im, label=L"\Im(a_2-a_1)")


In [ ]:
diff_norm = sqrt.(diff_re.^2 .+ diff_im.^2)
p1 = plot(t_list[1:end]*κ_2, diff_norm[1:end], label=L"\|a_2-a_1\|", yaxis=:log10, legend=:bottomright)

In [ ]:
using EasyFit

In [ ]:
diff_norm_log = log10.(diff_norm)

Lyap_coef = fitlinear(t_list[1:end]*κ_2, diff_norm_log[1:end])

In [ ]:
plot(t_list*κ_2, diff_norm, label="")
plot!(t_list[1:end]*κ_2, 10 .^(Lyap_coef.b .+ Lyap_coef.a * t_list[1:end]*κ_2), label="Lyapunov coef. = $(round(Lyap_coef.a, digits = 2))", linestyle=:dash, color=:red)
plot!(ylabel = L"\|a_2-a_1\|", xlabel = L"\kappa_2 t",
yaxis=:log10, 
legend=:bottomright,)

# We now try to use some julia packages to compute the Lyapunov exponent

In [ ]:
using ChaosTools

In [ ]:

function drive(t)
    return ϵ_d * cos(ω_d * t) #We consider a drive resonant with the mode b
end

ω_p=2ω_a-ω_d
function epsilon(t)
    return ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
end


temp_buf = zeros(n)
function two_photon_ODE!(du,u,p,t)
    ϕ_a = u[1]
    ϕ_b = u[2]
    Φ=u[3:n+2]
    ϕ_d_a = u[3+n]
    ϕ_d_b = u[4+n]
    Φ_d=u[5+n:end]
    
    du[1] = ϕ_d_a
    du[2] = ϕ_d_b
    du[3:n+2] = Φ_d
        
    du[3+n] = -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    du[4+n] = -ω_b^2*ϕ_b + ω_b * system_chain_coupling * sqrt(Z_b) * Φ[1] - ω_b *drive(t)*sqrt(2) + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    mul!(temp_buf, Mϕ_chain, Φ)
    @. temp_buf = -temp_buf
    temp_buf[1] += system_chain_coupling * sqrt(Z_b) * ϕ_b
    mul!(view(du, 5+n:length(du)), Mq_chain, temp_buf)
    return nothing
end

onsite_energies, hopping_energies, system_chain_coupling = chain_params
gq=vcat(- hopping_energies)
gϕ=vcat(hopping_energies)

Mq_chain=SymTridiagonal(onsite_energies, gq)
Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)

Mq_full=SymTridiagonal(vcat([ω_a,ω_b], onsite_energies), vcat([0,0],gq))

#Initial conditions
ϕ_a_0,q_a_0 = (0,0)

ϕ_b_0,q_b_0 = (0,0)


ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b), fill(0.0, n))
q_0 = vcat(q_a_0 * sqrt(Z_a),q_b_0 * sqrt(Z_b), fill(0.0, n))
ϕ_dot_0 = Mq_full*q_0
u_0 = vcat(ϕ_0,ϕ_dot_0)


#Run the simulation
tspan = (0.0, 20/κ_2) # Time span for the simulation
dt = 1 / (5 * maximum(vcat(ω_p,ω_a,ω_b,onsite_energies))) # Time step for the simulation
problem = ODEProblem(two_photon_ODE!, u_0, tspan)



In [ ]:

save_idxs=[1,2,n+3,n+4] # We only save the system DoFs (ϕ_a, ϕ_b) and not the chain DoFs (Φ)
sol = solve(problem, Tsit5(),
save_idxs=save_idxs,            # only store system DoFs
)

ϕ = [getindex.(sol.u,i) for i in 1:2]
ϕ_dot = [getindex.(sol.u,i+2) for i in 1:2]

ϕ_a = ϕ[1] * sqrt(Z_a)
q_a = ϕ_dot[1]/ω_a / sqrt(Z_a)

t_list = sol.t;

In [ ]:
Re_a = ϕ_a / sqrt(2*Z_a)
Im_a = q_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_a * cos(ω_a * t_list) + q_a*Z_a* sin(-ω_a * t_list)
q_rotating_a =@. -ϕ_a/Z_a * sin(-ω_a * t_list) + q_a* cos(ω_a * t_list)

Re_rotating_a = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a = q_rotating_a * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a,Im_a)
p2=plot(Re_rotating_a,Im_rotating_a)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("cat_state_chain.pdf")

In [ ]:
problem_bis = CoupledODEs(two_photon_ODE!, u_0, nothing)

In [ ]:
t_max = 20/κ_2
dt

In [ ]:
lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt, show_progress = true)

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.05:0.6pi)
ϵ_d_g2_ratio_list = 2:0.5:16

In [ ]:
using ProgressBars

In [ ]:
Lyap_coef = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
t_max=10/κ
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        ϕ_CL_a, _, q_CL_a, _, t_list_CL = two_photon_CL(t_max;
            circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0_i,κ_a=0,κ_b=κ,ϵ_d=ϵ_d_j,ω_d=ω_b_0),
            init_mode_a=(ϕ_a_0=0.001,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

        Re_rotating_a_CL_end =@. (ϕ_CL_a * cos(ω_a * t_list_CL) + q_CL_a*Z_a* sin(-ω_a * t_list_CL)) / sqrt(2*Z_a)
        Im_rotating_a_CL_end =@. (-ϕ_CL_a/Z_a * sin(-ω_a * t_list_CL) + q_CL_a* cos(ω_a * t_list_CL)) * sqrt(Z_a/2)

        ϕ_CL_a_2, _, q_CL_a_2, _, _ = two_photon_CL(t_max;
            circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0_i,κ_a=0,κ_b=κ,ϵ_d=ϵ_d_j,ω_d=ω_b_0),
            init_mode_a=(ϕ_a_0=0.00100001,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

        Re_rotating_a_CL_end_2 =@. (ϕ_CL_a_2 * cos(ω_a * t_list_CL) + q_CL_a_2*Z_a* sin(-ω_a * t_list_CL)) / sqrt(2*Z_a)
        Im_rotating_a_CL_end_2 =@. (-ϕ_CL_a_2/Z_a * sin(-ω_a * t_list_CL) + q_CL_a_2* cos(ω_a * t_list_CL)) * sqrt(Z_a/2)

        diff_re = Re_rotating_a_CL_end_2 - Re_rotating_a_CL_end
        diff_im = Im_rotating_a_CL_end_2 - Im_rotating_a_CL_end
        diff_norm = sqrt.(diff_re.^2 .+ diff_im.^2)
        diff_norm_log = log10.(diff_norm)
        
        Lyap_coef_i_j = fitlinear(t_list_CL*κ_2, diff_norm_log).a

        Lyap_coef[j,i] = Lyap_coef_i_j
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
α_sq= (ϵ_d_g2_ratio_list./2)
heatmap(g2_list/2pi,α_sq,Lyap_coef)
#plot!(g2_lis/2pi, (κ*1e3./(sqrt(2)*g2_lis)).^2,label="")
#plot!(g2_lis/2pi, (4*κ*1e3 ./g2_lis).^2,label="")
#plot!(g2_lis/2pi, (5.2*κ*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1]*1.3,size_px_car[2]*1.3),
    xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="Lyapunov coef.", title = L"κ_b t= "*"$(κ*t_max), "*L"n_{osc}="*"$(n)")

# We try to adapt the CL code with ChaosTools.jl

In [ ]:
ϵ_0 = 0.6*pi
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=0.75*g2

κ_2 = 4g2^2/κ;

In [ ]:
t_max = 10/κ

In [ ]:
function drive(ϵ_d,t)
    return ϵ_d * cos(ω_d * t) #We consider a drive resonant with the mode b
end

ω_p=2ω_a-ω_d
function epsilon(ϵ_0,t)
    return ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
end

# Definition of the second order ODE
function damped_two_photon(du, u, p, t)
    ϵ_d, ϵ_0, κ = p

    ϕ_a = u[1]
    ϕ_b = u[2]
    ϕ_p_a = u[3]
    ϕ_p_b = u[4]
    
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b
    du[3] = -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(ϵ_0,t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    du[4] = -ω_b^2*ϕ_b - ω_b *drive(ϵ_d,t)*sqrt(2) - 2 * κ * ϕ_p_b + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(ϵ_0,t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
end

# Initial conditions
ϕ_a_0, q_a_0 = (0,0)

ϕ_b_0, q_b_0 = (0,0)

ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b))
ϕ_dot_0 = vcat(q_a_0 * sqrt(Z_a)*ω_a,q_b_0 * sqrt(Z_b)*ω_b)
u_0 = vcat(ϕ_0,ϕ_dot_0)

tspan = (0.0, t_max) # Time span for the simulation


In [ ]:
dt = 1 / (5ω_a) # Time step for the simulation
prob = ODEProblem(damped_two_photon, u_0, tspan, (ϵ_d, ϵ_0))
sol = solve(prob, Tsit5(); dt=dt, dtmax=dt)

# Extract solution arrays
ϕ_dot_CL_a = [u[3] for u in sol.u]*sqrt(Z_a)
ϕ_CL_a = [u[1] for u in sol.u]*sqrt(Z_a)
q_CL_a = ϕ_dot_CL_a / C_a_inv

ϕ_dot_CL_b = [u[4] for u in sol.u]*sqrt(Z_b)
ϕ_CL_b = [u[2] for u in sol.u]*sqrt(Z_b)
q_CL_b = ϕ_dot_CL_b / C_b_inv;


In [ ]:
length(sol.t)

In [ ]:
p1, p2 = plot_mode(ϕ_CL_a, q_CL_a, sol.t, ω_a, Z_a)
layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)

In [ ]:
problem = CoupledODEs(damped_two_photon, u_0, (10*besselj(1,1)*E_J*φ_a^2*φ_b, 1))
X, t = trajectory(problem, t_max, Δt=dt)

In [ ]:
# Extract solution arrays
ϕ_dot_CL_a = getindex.(X,3)*sqrt(Z_a)
ϕ_CL_a = getindex.(X,1)*sqrt(Z_a)
q_CL_a = ϕ_dot_CL_a / C_a_inv

ϕ_dot_CL_b = getindex.(X,4)*sqrt(Z_b)
ϕ_CL_b = getindex.(X,2)*sqrt(Z_b)
q_CL_b = ϕ_dot_CL_b / C_b_inv;

In [ ]:
p1, p2 = plot_mode(ϕ_CL_a, q_CL_a, t, ω_a, Z_a)
layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)

In [ ]:
lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.05:0.6pi)
ϵ_d_g2_ratio_list = 2:0.5:16

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i

        problem = CoupledODEs(damped_two_photon, u_0, (ϵ_d_j, ϵ_0_i))

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
α_sq= (ϵ_d_g2_ratio_list./2)
heatmap(g2_list/2pi,α_sq,Lyap_coef_chaos)
#plot!(g2_lis/2pi, (κ*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ*1e3 ./g2_lis).^2,label="")
#plot!(g2_lis/2pi, (5.2*κ*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1]*1.3,size_px_car[2]*1.3),
    xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="\nLyapunov coef.", title = "Mean field "*L"κ_b t= "*"$(κ*t_max)",
    clims=(0,maximum(Lyap_coef_chaos)),
    right_margin=20mm)
#savefig("../../Figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
κ_b_list = 0.03:0.03:1
ϵ_0_list = vcat(0.5:.025:1,1:.05:0.6pi)

Lyap_coef_chaos = Array{Float64}(undef, length(κ_b_list), length(ϵ_0_list))
for (j,κ_b_i) in ProgressBar(enumerate(κ_b_list))
    t_max=10/κ_b_i
    for i in 1:length(ϵ_0_list)
        ϵ_0_i = ϵ_0_list[i]
        g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
        ϵ_d_j = 8*g2_max_i
        problem = CoupledODEs(damped_two_photon, u_0, (ϵ_d_j, ϵ_0_i, κ_b_i))
        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
heatmap(g2_list/2pi,κ_b_list/2pi,Lyap_coef_chaos)
#plot!(g2_lis/2pi, g2_lis/2/1e3/2pi, label=L"|\alpha|=4\kappa_b/g_2")
#plot!(g2_lis/2pi, 6g2_lis/1e3.-1.05)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\kappa_b/2\pi",
    title="Cat size "*L"\alpha^2=4",
    ylims=(minimum(κ_b_list/2pi),maximum(κ_b_list/2pi)),
    xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)))
#savefig("../../Figures/Two-photon/upper_bound_vs_kappa_b.pdf")

# We try to adapt the chain code with ChaosTools.jl

In [ ]:
n=3000

bath=Exp_decay
bath_params =(ω_0=50*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
onsite_energies, hopping_energies, system_chain_coupling = chain_params

gq=vcat(- hopping_energies)
gϕ=vcat(hopping_energies)

Mq_chain=SymTridiagonal(onsite_energies, gq)
Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)

Mq_full=SymTridiagonal(vcat([ω_a,ω_b], onsite_energies), vcat([0,0],gq));

In [ ]:
t_max = 10/κ

function drive(ϵ_d,t)
    return ϵ_d * cos(ω_d * t) #We consider a drive resonant with the mode b
end

ω_p=2ω_a-ω_d
function epsilon(ϵ_0,t)
    return ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
end

temp_buf = zeros(n)
function two_photon_ODE(du, u, p, t)
    ϵ_d, ϵ_0 = p

    ϕ_a = u[1]
    ϕ_b = u[2]
    Φ=u[3:n+2]
    ϕ_p_a = u[3+n]
    ϕ_p_b = u[4+n]
    Φ_p=u[5+n:end]

    
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b
    du[3:n+2] = Φ_p
    du[3+n] =  -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(ϵ_0,t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    du[4+n] = -ω_b^2*ϕ_b + ω_b * system_chain_coupling * sqrt(Z_b) * Φ[1] - ω_b *drive(ϵ_d, t)*sqrt(2) + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(ϵ_0,t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum

    mul!(temp_buf, Mϕ_chain, Φ)
    @. temp_buf = -temp_buf
    temp_buf[1] += system_chain_coupling * sqrt(Z_b) * ϕ_b
    mul!(view(du, 5+n:length(du)), Mq_chain, temp_buf)
    return nothing
end

# Initial conditions
ϕ_a_0, q_a_0 = (0,0)

ϕ_b_0, q_b_0 = (0,0)

ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b), fill(0.0, n))
q_0 = vcat(q_a_0 * sqrt(Z_a),q_b_0 * sqrt(Z_b), fill(0.0, n))
ϕ_dot_0 = Mq_full*q_0
u_0 = vcat(ϕ_0,ϕ_dot_0)

tspan = (0.0, t_max) # Time span for the simulation


In [ ]:
dt = 1 / (5ω_a) # Time step for the simulation
prob = ODEProblem(two_photon_ODE, u_0, tspan, (ϵ_d, ϵ_0))
save_idxs=[1,2,n+3,n+4] # We only save the system DoFs (ϕ_a, ϕ_b) and not the chain DoFs (Φ)
sol = solve(prob, Tsit5(); dt=dt, dtmax=dt,
        saveat=dt,                      # how often to save (or choose a coarser step)
        save_idxs=save_idxs,            # only store system DoFs
        save_everystep=false            # do not store intermediate solver steps
)

# Extract solution arrays
ϕ_dot_a = [u[3] for u in sol.u]*sqrt(Z_a)
ϕ_a = [u[1] for u in sol.u]*sqrt(Z_a)
q_a = ϕ_dot_a / C_a_inv

ϕ_dot_b = [u[4] for u in sol.u]*sqrt(Z_b)
ϕ_b = [u[2] for u in sol.u]*sqrt(Z_b)
q_b = ϕ_dot_b / C_b_inv;


In [ ]:
p1, p2 = plot_mode(ϕ_a, q_a, sol.t, ω_a, Z_a)
layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)

In [ ]:
problem = CoupledODEs(two_photon_ODE, u_0, (0.75*besselj(1,0.6pi)*E_J*φ_a^2*φ_b, 0.6pi))
X, t = trajectory(problem, t_max, Δt=dt)

In [ ]:
# Extract solution arrays
ϕ_dot_a = getindex.(X,n+3)*sqrt(Z_a)
ϕ_a = getindex.(X,1)*sqrt(Z_a)
q_a = ϕ_dot_a / C_a_inv

ϕ_dot_b = getindex.(X,n+4)*sqrt(Z_b)
ϕ_b = getindex.(X,2)*sqrt(Z_b)
q_b = ϕ_dot_b / C_b_inv;

In [ ]:
p1, p2 = plot_mode(ϕ_a, q_a, t, ω_a, Z_a)
layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)

In [ ]:
lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)

In [ ]:
t_max

In [ ]:
dt

In [ ]:
for i in 1:7
    @time lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
end

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.05:0.6pi)
ϵ_d_g2_ratio_list = 2:0.5:16

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i

        problem = CoupledODEs(damped_two_photon, u_0, (ϵ_d_j, ϵ_0_i))

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
α_sq= (ϵ_d_g2_ratio_list./2)
heatmap(g2_list/2pi,α_sq,Lyap_coef_chaos)
#plot!(g2_lis/2pi, (κ*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ*1e3 ./g2_lis).^2,label="")
#plot!(g2_lis/2pi, (5.2*κ*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1]*1.3,size_px_car[2]*1.3),
    xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="\nLyapunov coef.", title = "Chain simulation "*L"κ_b t= "*"$(κ*t_max)",
    clims=(0,maximum(Lyap_coef_chaos)),
    right_margin=20mm)
#savefig("../../Figures/Two-photon/heatmap_lyapunov_coef_chain.pdf")